# User Configuration

In [ ]:
# Set these paths to match your local environment before running
fmri_dir = ""
hands_annotations_dir = ""
cvat_annotations_dir = ""
motion_energy_template_path = ""

In [ ]:
results_outdir = 'results' # in root dir of the notebook
subjects = ['sub01', 'sub02','sub03','sub04','sub05','sub06']

In [ ]:
import os, sys

# Resolve utility paths relative to this notebook's directory
_nb_dir = os.path.abspath('')
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'speechmodeltutorial'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'utils_natcook'))

import os, glob, itertools, sys, pickle, ast
from os.path import join
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from nilearn import image
from nilearn.masking import apply_mask, unmask
from nilearn.image import index_img, threshold_img
from nilearn.plotting import plot_anat, view_img, plot_img

import cortex

from scipy.stats import pearsonr, zscore
from scipy import signal
from sklearn.decomposition import PCA
from itertools import product


from himalaya.backend import set_backend
from himalaya.kernel_ridge import MultipleKernelRidgeCV, ColumnKernelizer, Kernelizer
from himalaya.scoring import r2_score_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import check_cv
from voxelwise_tutorials.utils import generate_leave_one_run_out
from voxelwise_tutorials.delayer import Delayer
from voxelwise_tutorials.utils import explainable_variance


from statsmodels.stats.multitest import fdrcorrection


# ======================================= Custom imports =============================================

from interpdata import lanczosfun, lanczosinterp2D
from util import make_delayed
from ridge import bootstrap_ridge

from func_average_feature_weights_over_delays import average_feature_weights_over_delays
from voxelwise_permutation_test import voxelwise_permutation_test
from build_banded_ridge_pipeline import build_banded_ridge_pipeline

from FMRIPathConfig import FMRIPathConfig
from load_subject_data import load_subject_data
from average_repeated_runs import average_repeated_runs
from align_fmri_and_features import pad_runs_with_fixation, crop_annotations_to_match_scans, lanczos_downsample_annotations_to_TR, plot_feature_with_TRs
from format_features import fit_binaryencoder_across_runs, one_hot_encode_column, fit_multilabelbinarizer_across_runs, n_hot_encode_column, create_feature_spaces_dict
from compute_feature_counts import compute_feature_counts


In [ ]:
# ======================================= Input directories =============================================

# Use default input directory config
path_config = FMRIPathConfig(fmri_dir)
display('Path templates:', path_config.patterns)

# ======================================= Fixed parameters =============================================
annot_sampling_hz = 4 # the sampling rate of annotations from the .CSVs

# --- Runs numbers etc
# initial input data, used to read the data early on
init_session_numbers = [1,2]

init_run_numbers = [1,2,3,4,5,6]
init_train_run_numbers = [1,2,4,5]
init_test_run_numbers = [3,6]


# Run numbers once the data is processed, after averaging over repeats (3,6 for test runs)
all_run_numbers = [1,2,3,4,5]
train_run_numbers = [1,2,4,5]
test_run_number = 3


# --- TRs and timing info
TR = 2 # in seconds

# How much to pad each run's annotations to account for the fixarion cross time
start_fixation_len_secs = 16 # 20 secs minus 2 TRs that were dropped at preproc
end_fixation_len_secs = 22 # a couple of seconds more than needed, we will crop later to match scan length


# --- Other
n_jobs = -1
np.random.seed(42)

# =========================================================================================================
# ======================================= Analysis parameters =============================================
# =========================================================================================================
minimum_feat_count = 50


# --- Lanczos filter parameters
lanczos_window = 3 # Number of lobes for Lanczos kernel
lanczos_cutoff_mult = 1.0

# ----- Banded ridge parameters
choose_mask_modelled_voxels = 'cortical'
banded_ridge_params = dict(
# Hyperparameter search
    n_iter = 40,  # Number of random search iterations
    alphas = np.logspace(1, 20, 20),  # Range of alpha values to search
# Batch parameters for memory management
    n_targets_batch = 500,
    n_alphas_batch = 5,
    n_targets_batch_refit = 500,
)
delays = [1,2,3,4]


# --- Permutation test and FDR correction
kwargs_voxelwise_permutation_test = dict(
    chunk_length=10,
    n_permutations=1000,
    pval=0.05,
    return_null=True,
    seed=42,
    n_jobs=6)


fdrcorrection_alpha = 0.05


## Read annotations/features
These are shared across subjects

In [ ]:
# OUTPUT VARS
runs_feats = {run_number: None for run_number in all_run_numbers} # output dict, keys=all_run_numbers, each eventually with shape (n_TRs, n_feats)
feats_legend = [] # list of len=n_feart. list of str to allow me to keep track of what each feature index represents

#### HANDS
**Note:** the rest of the script will assume z-scoring was done at this step

In [ ]:
hands_annot = {}
for run_number in all_run_numbers:
    print('Warning: Filling NAN with zeros, remove if needed')
    hands_annot[run_number] = pd.read_csv(join(hands_annotations_dir, f'run{run_number}_annotation_grasping_similarity_matrix.csv')).fillna(0)

**Hand landmark features: Linear combination of primitives**


z-scoring features within each run (over time)


In [ ]:
for hand in ['left', 'right']:

    # get the name of the landmarks columns
    features_column_names = [f'hands_{hand}_prim_{i}' for i in range(1, 21)]

    # -- 1) z-score features within each run
    zscored_features_noPCA = {} # intermediate output dict where we will store z-scored features for each run.
    
    for run_number in all_run_numbers:
        # Get the raw features from this run's annot dataframe
        features = hands_annot[run_number][features_column_names].values
        
        # Scale the features over time
        scaled_features = zscore(features, axis=0)

        # Cant concatenate on the first iteration, when run_feats[run_number] is still an empty list
        if runs_feats[run_number] is None:
            runs_feats[run_number] = scaled_features
        else: # If run_feats[run_number] is not empty, concatenate on axis 1
            runs_feats[run_number] = np.concatenate([runs_feats[run_number], scaled_features], axis=1)

    # 2) Update feats_legend with the names of the z-scored features
    feats_legend.extend(features_column_names)

**Binary feature indicating whether hand was on/off screen**

This allows to interpret coordinates of all 0s as the absence of hand, rather a hand with all nodes at [0,0,0]

Binary variable, no scaling done

In [ ]:
features_column_names = ['hands_left_on', 'hands_right_on']
for run_number in all_run_numbers:
    # Get the raw features from this run's annot dataframe
    features = hands_annot[run_number][features_column_names].values
    
    runs_feats[run_number] = np.concatenate([runs_feats[run_number], features], axis=1)

# Append the feature labels to the feats_legend list. Done outside of loop: one list will refer to all runs.
feats_legend.extend(features_column_names)

#### CVAT

In [ ]:
# read CVAT CSVs
labels_to_exclude = {'r_hand', 'l_hand', 'b_hands'}

def replace_in_string(val):
    if isinstance(val, str):
        return np.nan if val in labels_to_exclude else val
    return val

def replace_in_list(lst):
    return [item for item in lst if item not in labels_to_exclude]


CVAT_annot = {}
for run_number in all_run_numbers:
    df = pd.read_csv(join(cvat_annotations_dir, f'run{run_number}_annotations_clean_v5.csv'))

    # Convert string representations of lists to actual lists
    df['objects_on_screen'] = df['objects_on_screen'].apply(ast.literal_eval)
    df['unique_objects_on_screen'] = df['unique_objects_on_screen'].apply(ast.literal_eval)

    # Replace in list-based columns
    df['objects_on_screen'] = df['objects_on_screen'].apply(replace_in_list)
    df['unique_objects_on_screen'] = df['unique_objects_on_screen'].apply(replace_in_list)

    # Replace in string-based columns
    for col in df.columns:
        if col not in ['objects_on_screen', 'unique_objects_on_screen']:
            df[col] = df[col].apply(replace_in_string)

    # Sanity check with hands_annot
    assert len(df) == len(hands_annot[run_number]), f'Hands and CVATs annot CSVs dont match in length for run={run_number}'

    # Store the cleaned df
    CVAT_annot[run_number] = df


**1) One hot encoding vectors for actions; N-hot for targets (targets + tools) (no z-scoring)**

In [ ]:
# --- Actions: unchanged ---
print('Processing actions')
lb_action = fit_binaryencoder_across_runs(CVAT_annot, 'action')
print(f'{len(lb_action.classes_)} unique labels found : \n{lb_action.classes_}')

for run_number in all_run_numbers:
    onehot = one_hot_encode_column(CVAT_annot[run_number], lb_action, 'action')
    runs_feats[run_number] = np.concatenate([runs_feats[run_number], onehot], axis=1)

feats_legend.extend([f'action_{i}' for i in lb_action.classes_])


# --- Target + Tool: merged into a single multihot vector ---
print('Processing target objects')

# Build a combined vocabulary across both columns and all runs
all_target_tool_labels = sorted(set(
    label
    for col in ['target', 'tool']
    for run_number in all_run_numbers
    for label in CVAT_annot[run_number][col].dropna().unique()
))
print(f'{len(all_target_tool_labels)} unique labels found : \n{all_target_tool_labels}')

label_to_idx = {label: idx for idx, label in enumerate(all_target_tool_labels)}
n_labels = len(all_target_tool_labels)

for run_number in all_run_numbers:
    df = CVAT_annot[run_number]
    n_frames = len(df)
    multihot = np.zeros((n_frames, n_labels), dtype=float)

    for col in ['target', 'tool']:
        for frame_idx, label in enumerate(df[col]):
            if pd.notna(label) and label in label_to_idx:
                multihot[frame_idx, label_to_idx[label]] = 1.0

    runs_feats[run_number] = np.concatenate([runs_feats[run_number], multihot], axis=1)

feats_legend.extend([f'target_{label}' for label in all_target_tool_labels])

**2) N-hot encoding vector for passive objects on screen**

Prefix will be object_... in feats_legend

In [ ]:
colname = 'unique_objects_on_screen'
print(f'Processing "{colname}" column')

mlb = fit_multilabelbinarizer_across_runs(CVAT_annot, colname)
print(f'{len(mlb.classes_)} unique labels found:\n{mlb.classes_}')

for run_number in all_run_numbers:
    n_hot = n_hot_encode_column(CVAT_annot[run_number], mlb, colname)
    
    # Zero out target/tool objects from the passive objects encoding at frames where they are being manipulated
    for frame_idx, row in CVAT_annot[run_number].iterrows():
        target_obj = row['target']
        tool_obj = row['tool']

        for obj in [target_obj, tool_obj]:
            if pd.notna(obj) and obj in mlb.classes_:
                obj_col_idx = list(mlb.classes_).index(obj)
                n_hot[frame_idx, obj_col_idx] = 0
    
    runs_feats[run_number] = np.concatenate([runs_feats[run_number], n_hot], axis=1)

feature_names = [f'object_{obj}' for obj in mlb.classes_]
feats_legend.extend(feature_names)

**MOTION FEATURES**

In [ ]:
for run_number in all_run_numbers:

    # read motion energy data    
    motion = np.load(motion_energy_template_path.format(str(run_number)))

    # crop the end of motion energy data to match length of the rest of the annots
    motion = motion[:runs_feats[run_number].shape[0], :]
    assert runs_feats[run_number].shape[0] == motion.shape[0]

    # average over all the motion energy features:
    motion = motion.mean(axis=1)

    # zscore over time (the only dimension remaining anyway)
    motion = zscore(motion)

    # reshape to 2D for concatenation with the rest of the features
    motion = motion.reshape(-1, 1)  # or motion[:, np.newaxis]

    # concatenate to existing features
    runs_feats[run_number] = np.concatenate([runs_feats[run_number], motion], axis=1)


# Update feature names
feats_legend.append('motion_energy')


#### Drop features that occur less than `minimum_feat_count` times in the train runs

with the count here refers to a single frame of tha annotations, not a continous chunk.


In [ ]:
# === STEP 1: Compute feature occurrence counts ===
features_counts, features_freqs_sorted = compute_feature_counts(
    runs_feats=runs_feats,
    train_run_numbers=train_run_numbers,
    feats_legend=feats_legend,
    presence_threshold=0
)

# === STEP 2: Determine features to keep or drop ===
keep_indices = np.where(features_counts >= minimum_feat_count)[0]
drop_indices = np.where(features_counts < minimum_feat_count)[0]

# Safety check: ensure at least one feature remains
if len(keep_indices) == 0:
    raise ValueError("All features were dropped. Consider lowering `minimum_feat_count`.")

feats_legend_kept = [feats_legend[i] for i in keep_indices]
feats_legend_dropped = [feats_legend[i] for i in drop_indices]

# === STEP 3: Log sorted feature frequencies ===
kept_sorted = sorted(zip(keep_indices, [features_counts[i] for i in keep_indices]), key=lambda x: -x[1])
dropped_sorted = sorted(zip(drop_indices, [features_counts[i] for i in drop_indices]), key=lambda x: x[1])

print(f"\nKept {len(kept_sorted)} features (>= {minimum_feat_count} occurrences):")
for i, count in kept_sorted:
    print(f"  {feats_legend[i]}: {count}")

print(f"\nDropped {len(dropped_sorted)} features (< {minimum_feat_count} occurrences):")
for i, count in dropped_sorted:
    print(f"  {feats_legend[i]}: {count}")

# === STEP 4: Filter features in all runs ===
runs_feats_filtered = {
    run_number: feats[:, keep_indices]
    for run_number, feats in runs_feats.items()
}

# === STEP 5: Overwrite original structures ===
runs_feats = runs_feats_filtered
feats_legend = feats_legend_kept


**When all is done, check shapes are good**

In [ ]:
# make sure shapes are good
print('Features extracted: \n--------------------')
[print(i) for i in feats_legend]

for run_number in all_run_numbers:
    assert runs_feats[run_number].shape[1] == len(feats_legend), f'Shape mismatch {runs_feats[run_number].shape[1]} = {len(feats_legend)}'

print('-------------------')
print('num feats for each run: ',len(runs_feats[1][1]))
print('len(feats_legend) =', len(feats_legend))

# plot design matrix of run 1
fig, ax = plt.subplots(figsize=(10,15))
sns.heatmap(runs_feats[1].T, vmin=0, vmax=1, yticklabels=feats_legend, ax=ax)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8);


#### Add the padding for fixation cross at the beginning and the end
For each feature, we add 20s at the beginning and the end to account for the 20secs of fixation that padded each video.

Since at preproc we drop 2 TRs at the beginning of each scan, we actually add 16secs = 8TRs at the beginning.

In [ ]:
runs_feats = pad_runs_with_fixation(
    runs_feats=runs_feats,
    run_numbers=all_run_numbers,
    start_fixation_len_secs=start_fixation_len_secs,
    end_fixation_len_secs=end_fixation_len_secs,
    annot_sampling_hz=annot_sampling_hz
)

#### Create `feature_spaces` dictionary used in the banded ridge

In [ ]:
# Define categories and their identifying prefixes
categories = [
    ('hands_', 'hand_features'),
    ('action_', 'action_features'),
    ('target_', 'target_features'),
    ('object_', 'object_features'),
    ('motion_energy', 'motion_features'),
]

feature_spaces = create_feature_spaces_dict(feats_legend, categories)

## Save all features etc to file.

In [ ]:
import pickle

with open('runs_feats.p', 'wb') as f:
    pickle.dump(runs_feats, f)
with open('feats_legend.p', 'wb') as f:
    pickle.dump(feats_legend, f)
with open('feature_spaces.p', 'wb') as f:
    pickle.dump(feature_spaces, f)

## Read and Process fMRI data, then fit the models

In [ ]:
for subj in subjects:
        
    print(f'\n\n-------------------\nSubject : {subj}\n')

    subj_outdir = join(results_outdir, subj)
    if not os.path.exists(subj_outdir):
        os.makedirs(subj_outdir)


    ridge_outpath = join(subj_outdir, f'{subj}_ridge_results.p')
    
    if os.path.exists(ridge_outpath):
        print(f'{subj}: Already done, skipping.')
        continue

    else:

        # ======================================================================================= #
        print('\n(1) Loading all runs')
        init_runs_data, brainmask_img = load_subject_data(subj, path_config, init_session_numbers, init_run_numbers)

        # ======================================================================================= #
        print('\n(2) Averaging over repeats')
        runs_data = average_repeated_runs(init_runs_data, init_train_run_numbers, init_test_run_numbers)

        # rename 'test_avg' key to 3  (=test_run_number)
        print(runs_data.keys())
        runs_data[test_run_number] = runs_data.pop('test_avg')
        print(runs_data.keys())

        # ======================================================================================= #
        print('\n(3) Aligning features to runs')
        print('   Crop annotations to match run lengths')
        runs_feats_subj = crop_annotations_to_match_scans( # Saving the modified runs_feats as runs_feats_subj so that the global version doesnt get overwritten on the next subj iteration
            runs_feats=runs_feats,
            runs_data=runs_data,
            annot_sampling_hz=annot_sampling_hz,
            TR=TR,
            verbose=True)

                

        print('   Aligning sampling rates of features to runs using Lanczos filtering')
        downsampled_feats = lanczos_downsample_annotations_to_TR(
                                            runs_feats_subj,
                                            runs_data,
                                            annot_sampling_hz,
                                            TR,
                                            lanczos_window,
                                            lanczos_cutoff_mult,
                                            plot_result_sample=False) 
        # ======================================================================================= #
        # Summary of variable shapes

        print("\n(5) Feature and fMRI Data Shapes by Run\n" + "-" * 50)
        for run_number in all_run_numbers:
            print(f"Run {run_number}")
            print(f"  {'runs_feats_subj':<20} shape: {runs_feats_subj[run_number].shape}")
            print(f"  {'downsampled_feats':<20} shape: {downsampled_feats[run_number].shape}")
            print(f"  {'runs_data':<20} shape: {runs_data[run_number].shape}")
            print("-" * 50)

        # ======================================================================================= #
        print('\n(6) Movement covariates: nothin done.')
        # If wanted to add head movement covariates here.



        # ======================================================================================= #
        print('\n(7) Setting up Banded Ridge Regression')

        # Set backend (use GPU if available)
        backend = set_backend("torch_cuda", on_error="warn")

        # ======================================================================================= #
        # Create cross validation slits using generate_leave_one_run_out().
        print('   Creating cross-validation splits')

        # Calculate the number of samples in each training run
        train_run_lengths = [runs_data[run_number].shape[0] for run_number in train_run_numbers]
        print(f"Training run lengths: {dict(zip(train_run_numbers, train_run_lengths))}")

        # Create run_onsets: cumulative sum starting from 0
        run_onsets = np.concatenate([[0], np.cumsum(train_run_lengths[:-1])])
        print(f"Run onsets: {run_onsets}")


        # Create cross-validation splits
        n_samples_train = sum(train_run_lengths)
        cv = generate_leave_one_run_out(n_samples_train, run_onsets)
        cv = check_cv(cv)


        # ======================================================================================= #
        # Prepare train/test data and features
        print('   Preparing data for banded ridge')

        # Stack training data and features
        train_data = np.vstack([runs_data[run_number] for run_number in train_run_numbers])
        test_data = runs_data[test_run_number]

        train_feats = np.vstack([downsampled_feats[run_number] for run_number in train_run_numbers])
        test_feats = downsampled_feats[test_run_number]


        # ======================================================================================= #
        # Mask out unwanted voxels (e.g. using cortical mask). 
        # Reminder: if loading a mask using cortex.get_cortical_mask, remember to transpose the array so it matches the orientation of nilearn data
        print('   Creating voxel mask (optional)')

        # load cortical mask
        if choose_mask_modelled_voxels == 'cortical':
            cortical_mask = np.load(path_config.get_cortical_mask_path(subj))

        elif choose_mask_modelled_voxels == 'thick':
            cortical_mask = cortex.get_cortical_mask(subj, f'{subj}_transform', type='thick')
            cortical_mask = cortical_mask.T # go from pycortex format to nilearn format

        elif choose_mask_modelled_voxels == 'None':
            mask_modelled_voxels = np.ones(train_data.shape[1], dtype=bool) # Use all voxels (no masking) — get number of voxels from training data shape
            mask_modelled_voxels_img = unmask(mask_modelled_voxels, brainmask_img) # reconstruct the 3d mask for use later, and for viz here

        else:
            raise ValueError(f"Invalid mask choice: {choose_mask_modelled_voxels}")


        if choose_mask_modelled_voxels != 'None': # Only run this if we're not in the 'None' branch (already done above)
            cortical_mask_img = image.new_img_like(brainmask_img, cortical_mask)

            mask_modelled_voxels = apply_mask(cortical_mask_img, brainmask_img).astype('bool') # flatten the cortical mask, to match the shape of the data
            mask_modelled_voxels_img = unmask(mask_modelled_voxels, brainmask_img) # reconstruct the 3d mask for use later, and for viz here
                

        print('Mask shape :', mask_modelled_voxels.shape)
        print(f'Fitting on {np.sum(mask_modelled_voxels):.0f} voxels')


        # Visualize the mask, saves to file
        mask_figure_outpath = join(subj_outdir, f'{subj}_modeled_vox_mask.jpg')
        anat_img = image.load_img(path_config.get_anat_path(subj)) # to plot as background
        fig_mask = plot_img(mask_modelled_voxels_img, 
                            bg_img=anat_img, 
                            threshold=1e-5, # this makes 0 voxels transparent here
                            cmap='autumn',
                            title=f'Modeled voxels. Mask={choose_mask_modelled_voxels}')
        fig_mask.savefig(mask_figure_outpath)
        fig_mask.close()

        # ======================================================================================= #
        # Calculate explainable variance here. We need to do it here because we need mask_modelled_voxels.
        
        # Before deleting init_runs_data let's calculate the explainable variance
        ev_data = []
        for run_key in init_runs_data.keys():
            for nested_key in init_test_run_numbers:
                if nested_key in init_runs_data[run_key]:
                    ev_data.append(init_runs_data[run_key][nested_key][:, mask_modelled_voxels])

        ev_data = np.array(ev_data) # this now has shape (n_repeated_test_runs, n_TRs, n_voxels)

        ev = explainable_variance(ev_data)
        ev_img = unmask(ev, mask_modelled_voxels_img) # reconstruct it to original volumetric space

        del init_runs_data, ev_data # we can now free up memory
        # ======================================================================================= #
        # Build banded ridge pipeline
        pipeline, column_kernelizer = build_banded_ridge_pipeline(feature_spaces, banded_ridge_params, cv, delays)


        # ======================================================================================= #
        # Fit the banded ridge model
        print('   Fitting banded ridge model')
        print("   This may take several minutes...")

        # Fit the model
        pipeline.fit(train_feats, train_data[:, mask_modelled_voxels])

        # Score on test data
        scores_mask = pipeline.score(test_feats, test_data[:, mask_modelled_voxels])
        scores_mask = backend.to_numpy(scores_mask)

        # Extend scores to all voxels (if using masking)
        n_voxels = train_data.shape[1]
        scores = np.zeros(n_voxels)
        scores[mask_modelled_voxels] = scores_mask

        print(f"Banded ridge scores - Min: {np.min(scores):.3f}, Max: {np.max(scores):.3f}, Mean: {np.mean(scores):.3f}")


        # ======================================================================================= #
        # Get split predictions for each feature space
        print('   Computing feature space contributions')

        # Get predictions split by feature space
        Y_test_pred_split = pipeline.predict(test_feats, split=True)
        print(f"Split predictions shape: {Y_test_pred_split.shape}")

        # Compute split scores
        split_scores_mask = r2_score_split(test_data[:, mask_modelled_voxels], Y_test_pred_split)

        # Extend to all voxels
        n_kernels = split_scores_mask.shape[0]
        split_scores = np.zeros((n_kernels, n_voxels))
        split_scores[:, mask_modelled_voxels] = backend.to_numpy(split_scores_mask)

        print("Feature space contributions computed")
        for i, name in enumerate(feature_spaces.keys()):
            print(f"  {name}: Mean score = {np.mean(split_scores[i]):.3f}")



        # ======================================================================================= #
        # Run voxelwise permutation test

        # First, generate prediction
        pred = pipeline.predict(test_feats) # no need to mask anything here, the model was trained on masked data, predicted data has "masked" shape already

        # Second, voxelwise_permutation_test(). Mask out the test_data using mask_modelled_voxels.
        voxcorrs_true, sig_mask_uncorrected, pval_map, null_distrib = voxelwise_permutation_test(pred, test_data[:, mask_modelled_voxels], **kwargs_voxelwise_permutation_test)


        # FDR correction on the modeled voxels only
        sig_mask_fdr, pvals_fdr_corrected = fdrcorrection(pval_map, alpha=fdrcorrection_alpha)
        print(f"FDR-corrected significant voxels: {np.sum(sig_mask_fdr)} / {len(pval_map)}")


        # ======================================================================================= #
        # Save results
        print('\n(8) Saving banded ridge results')

        # Create results dictionary
        banded_ridge_results = {
            'pipeline': pipeline,
            'scores': scores,
            'split_scores': split_scores,
            'feats_legend' : feats_legend,
            'feature_spaces': feature_spaces,
            'feature_space_names': list(feature_spaces.keys()),
            'solver_params': banded_ridge_params,
            'mask_modelled_voxels': mask_modelled_voxels,
            'permutation_results': {
                            'voxcorrs_true' : voxcorrs_true, 
                            'sig_mask_uncorrected' : sig_mask_uncorrected, 
                            'pval_map' : pval_map, 
                            'null_distrib' : null_distrib},
            'fdr_results': {
                            'sig_mask_fdr' : sig_mask_fdr, 
                            'pvals_fdr_corrected' : pvals_fdr_corrected},
            'delays': delays,
            'explainable_variance_img' : ev_img

        }

        with open(ridge_outpath, 'wb') as f:
            pickle.dump(banded_ridge_results, f)

        # ======================================================================================= #

        print(f'{subj}: Done.')
        # Clear large variables to free up memory for next subject
        del runs_data, runs_feats_subj, downsampled_feats
        del train_data, test_data, train_feats, test_feats
        del pipeline, pred, scores_mask, split_scores_mask
        del voxcorrs_true, pval_map, null_distrib
        del mask_modelled_voxels_img, anat_img

